## This is the code to train the model and acquire influence for Class Imbalance Experiment

**Default Code**:   
The current default code is a runnable sample. It runs on the synthetic dataset generated with sklearn's make_classification function. The default version code provides the synthetic dataset with 16500 samples and 10 features in total. The separation is set to 1.5 to make sure the dataset is distinguishable by the model. All features are set to be informative to ensure they are of equal importance. The dataset has only two labels, so it is a binary classification problem. The default setting will then generate the training set and test set from the pool. The default training sample size is 7500, and the test size is 500. The number of features is set to 10. Currently, the number of samples within each class is balanced. This can be changed by modifying the cur_ratio. The model in default will be a Simple FeedForward Neural Network constructed by TensorFlow. The Influence Estimation methods we provide by default are the Influence Function and TracIn. If you simply press 'play', the default code will generate ranked influence lists for both Influence Function and TracIn with respect to the above mentioned setting in the root directory. The result lists could then be fed into other analyses.

**By default, the only thing changing here is how we pre-process the datasets. Basically, the number of samples within each class will be changed based on the choice of cur_ratio. This shall decide the level of imbalance within the train and test sets.** For the other common information, Please refer to the base code for more detailed explanation.

**Guideline**:  
Read in / Construct Datasets -> **Choose the Current Imbalance Ratio** -> Pre-processing -> Model Training -> Influence Estimation -> Store the Ranked Influence lists -> **Change the Imbalance Ratio and Repeat all the process** -> ... -> **After all the training and estimation, feed the results into the analysis code**  (Remember to change the file name in the last block to save lists in different settings.)

# Import Area

Here is the area to place all the import codes. You don't need to change here unless you want to customise in later sections.

In [57]:
import tensorflow as tf
import keras
from keras.utils import to_categorical
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [58]:
from keras import Sequential
from keras.layers import Dense, BatchNormalization, Dropout
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam

In [59]:
from deel.influenciae.common import InfluenceModel, ExactIHVP
from deel.influenciae.influence import FirstOrderInfluenceCalculator
from deel.influenciae.utils import ORDER
from deel.influenciae.trac_in import TracIn

In [60]:
import random
from keras.optimizers import SGD

In [61]:
from sklearn.metrics import pairwise_distances
from sklearn.manifold import MDS
import seaborn as sns
import matplotlib.pyplot as plt

In [62]:
from sklearn.datasets import make_classification

Train_Size: 1000, 2000, 4000, 8000, 16000  
Feature_Size: 10, 20, 40, 80, 160

# Dataset Construction Area

**You can use any dataset you wish here, either regression or classification. But in default, since we are using influenciae's IF and TC method, make sure they are split into train and test sets, and then stored as tensorflow dataset format. If you only want to change the dataset, you can only change the code in the first two blocks to read in/ generate your own dataset. But remember to have features X and target y before going to the third block. Also, if you wish to everything on your own, remember to add id inside the dataset.** Since our default code is for classification, the regression might need a lot of changes in all the following sections.


**Input**: Dataset chosen(Usually in Features X and Target y format)  
**Output**: Tensorflow format Train and Test Set  
**Guideline**: Input -> Turn into Dataframe and add ID -> Pre-Processing (Change the balance of each class) -> Change the format to Tensorflow -> Output

The default code now produces a synthetic dataset with 16500 pool, 160 features with binary classification problems. The later options will turn that into a 10 features, 7500 train set and 500 test set sample. The current ratio is (5,5), which means the class is balanced right now. Both sets will then be turned into TensorFlow format and will wait for training.

**The most important thing in this code is changing the cur_ratio. In this experiment, all the other things are fixed, but the balance between each class is changing to test how the imbalance of class affect the influence estimation.**

1. Set your default setting here. train_pool + test_size = Total Dataset Size. Ratios determine the class imbalance level. Sep to make sure the dataset is distinguishable.

In [63]:
train_pool = 16000
test_size = 500
n_features=10
seed=42
ratios = [(9,1), (8,2), (7,3), (6,4), (5,5)]
sep = 1.5

2. Construct the Synthetic Dataset with Make Classification here. **Could replace this with other datasets with X and y.**

In [64]:
total_samples = train_pool + test_size

X, y = make_classification(n_samples=total_samples,
                           n_features=n_features,
                           n_informative=n_features,
                           n_redundant=0,
                           n_repeated=0,
                           n_classes=2,
                           class_sep=sep,
                           random_state=seed)

In [65]:
k= 0

3. Turn the X and y into dataframe for easy further processing. Add ID column to easy retrieve samples within IF/TC.

In [66]:
df = pd.DataFrame(X, columns=[f'feature_{i+1}' for i in range(n_features+k)])
df['label'] = y
df['id'] = np.arange(1, len(df) + 1)
print(df)

       feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0      -2.042923   2.447225   1.459419  -4.895205  -1.931741   2.439100   
1      -3.021719  -2.189454   1.594643  -0.936749   6.580163  -0.856511   
2       1.744477  -1.747769  -1.803517  -3.028068  -0.517037   3.262006   
3      -0.372644   3.341254   1.849793   0.905480   0.359212   1.766193   
4      -1.972171  -3.053477   1.529830   1.741359   0.858712   1.675452   
...          ...        ...        ...        ...        ...        ...   
16495  -1.318844  -1.342788   0.624967  -0.463707  -0.735329   1.971186   
16496  -3.063709  -0.202935  -0.534888  -6.109804  -0.977436   1.353489   
16497   0.408280  -2.252040   2.995770  -0.442872  -0.904981   2.307891   
16498  -0.117513   3.516412  -3.900624  -2.741435  -4.439399  -0.522876   
16499   3.901795  -1.910619  -0.779033  -0.536239  -1.471359   0.685229   

       feature_7  feature_8  feature_9  feature_10  label     id  
0      -0.694749  -3.070256  -3.

4. **The most important thing in this experiment comes here.** First of all, we choose the exact dataset size during the training by setting value of exact_size. Then, the imbalance level can be set by changing the cur_ratio = ratios[4]. Then, the rest of the code will construct our desired dataset based on the ratio from the data pool. 500 samples will be chosen from them to be the test set. The test set is always set to be balanced so that the overall influence of each training sample will not be biased. 

In [67]:
exact_size = 8500

In [68]:
train_size = exact_size - test_size

In [69]:
cur_ratio = ratios[0]
print(cur_ratio)

(9, 1)


In [70]:
major, minor = cur_ratio

In [71]:
df0 = df[df.label == 0]  
df1 = df[df.label == 1] 

In [72]:
# t0 = int(exact_size * major / (major + minor))
# t1 = exact_size - t0 
# print(t0,t1)

In [73]:
t0 = int(major * train_size / (major + minor) + test_size // 2)
t1 = int(minor * train_size / (major + minor)+ test_size // 2)
print(t0,t1)

7450 1050


In [80]:
s0 = df0.sample(n=t0, random_state=seed)
s1 = df1.sample(n=t1, random_state=seed)

In [81]:
df = pd.concat([s0, s1], axis=0).sample(frac=1.0, random_state=seed).reset_index(drop=True)

In [82]:
print(df)
print(df["label"].value_counts())

      feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0      2.000551   6.107697   0.922888  -0.645629  -2.044902   6.067214   
1     -1.482890   3.501781   1.723413  -0.987130   4.693062   3.417544   
2     -1.461155   4.709173   1.276742   2.226696   1.542537   5.980747   
3      2.349369   2.618102   0.891645  -1.675392   0.253586  -0.248629   
4     -3.087668   2.723457   0.627227   3.464238  -0.751775   0.861041   
...         ...        ...        ...        ...        ...        ...   
8495  -0.392552  -4.158898   4.090073   1.949615  -0.431500  -0.218027   
8496   3.581924  -1.483711  -0.054275  -2.953603  -1.952614  -2.313367   
8497   2.125435  -1.880934  -2.997253  -3.589638  -2.528225   0.119474   
8498  -1.806808  -0.704839   1.698431  -3.602001   0.844647   1.772143   
8499  -1.416175  -0.981882  -0.853378  -0.154625  -5.670463  -1.923880   

      feature_7  feature_8  feature_9  feature_10  label     id  
0     -1.856265  -2.283272  -2.149731   -5.91

5. **Another important thing in this experiment comes here.** Here, we shall store the corresponding dataframe of each label and id so that we can map the minority group with their influence. That is to say, we need to know which group does each sample belong to.

In [83]:
id_label_df = df[["id", "label"]].copy()
print(id_label_df)
id_label_df.to_csv("Class_label_9_1_Run1.csv",index = False)

         id  label
0      5018      1
1     10999      1
2      3146      1
3      9147      0
4     12955      1
...     ...    ...
8495  13382      0
8496  13339      0
8497  15658      0
8498  12979      0
8499   9910      1

[8500 rows x 2 columns]


6. Cut the dataset into train set and test set. They have equal number to make sure each contribute equally to the overall influence.

In [84]:
n0 = test_size //2
n1 = test_size - n0

In [85]:
g = df.groupby("label", group_keys=False)
test_df = pd.concat([
    g.get_group(0).sample(n=n0, random_state=42, replace=False),
    g.get_group(1).sample(n=n1, random_state=42, replace=False),
]).sample(frac=1, random_state=42)

train_df = df.drop(test_df.index).reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

In [86]:
print(test_df.groupby("label").get_group(0))
print(test_df["label"].value_counts())

     feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
1     0.834430   0.463094   1.273167  -1.544390   0.188683  -1.068960   
3     0.607505   1.762207  -0.338045  -1.004661  -1.166495  -0.464131   
4     1.942862   0.143290  -0.346069   3.511778   2.656194  -3.307865   
7     6.311960   1.746709   1.541404   1.978819  -1.092200  -2.673320   
8     0.916350  -0.132366  -2.954575  -2.036922  -0.780542   2.473928   
..         ...        ...        ...        ...        ...        ...   
492  -2.945985   1.151929   2.848188  -4.928831   1.101044   4.751279   
493  -1.706920   0.867694  -0.438359  -0.844137   1.334499  -2.068917   
494   1.130985   3.047467  -0.718695   0.831319  -0.792173  -2.747026   
495   1.451834   1.841968  -5.509639  -1.650674  -1.608023  -0.265070   
499   0.139369  -0.459396   0.936389  -4.317837  -4.288256  -2.819963   

     feature_7  feature_8  feature_9  feature_10  label     id  
1    -0.365911  -0.208600   2.786547   -1.965793      0   

In [87]:
print(train_df["label"].value_counts())

label
0    5600
1    2400
Name: count, dtype: int64


In [88]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]
IDs = X_train["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_train = X_train.drop(columns=["id"]).values.astype(np.float32)
X_train = np.hstack((X_train, IDs))
y_train = to_categorical(y_train.values,num_classes=2)

print(X_train)

[[ 2.0005512e+00  6.1076975e+00  9.2288822e-01 ... -2.1497307e+00
  -5.9146123e+00  5.0180000e-07]
 [-1.4828904e+00  3.5017807e+00  1.7234126e+00 ... -1.7308999e+00
  -1.5941935e+00  1.0999000e-06]
 [-1.4611551e+00  4.7091732e+00  1.2767420e+00 ... -3.4669182e-01
  -1.2326162e+00  3.1459999e-07]
 ...
 [ 2.1254346e+00 -1.8809341e+00 -2.9972529e+00 ...  2.2385364e+00
   5.0112934e+00  1.5657999e-06]
 [-1.8068081e+00 -7.0483923e-01  1.6984305e+00 ...  1.1623163e+00
   3.8293433e-01  1.2979000e-06]
 [-1.4161745e+00 -9.8188233e-01 -8.5337770e-01 ...  4.7182436e+00
   2.0389636e+00  9.9099998e-07]]


In [89]:
X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]
IDs = X_test["id"].values.reshape(-1, 1).astype(np.float32)
IDs = IDs  / 1e10

X_test = X_test.drop(columns=["id"]).values.astype(np.float32)
X_test = np.hstack((X_test, IDs))
y_test = to_categorical(y_test.values,num_classes=2)

print(X_test)

[[-6.9917929e-01 -2.1819067e+00 -1.7673473e+00 ...  1.4914845e+00
  -2.9403925e+00  2.5380001e-07]
 [ 8.3442974e-01  4.6309391e-01  1.2731669e+00 ...  2.7865467e+00
  -1.9657933e+00  6.4749997e-07]
 [ 1.7611679e+00 -1.0732651e+00 -2.2256107e+00 ...  1.8814717e-01
   1.3768823e-01  6.0069999e-07]
 ...
 [-1.0141234e+00  2.8594145e-01 -7.5073397e-01 ... -1.2288723e+00
  -3.4612174e+00  1.4724000e-06]
 [-1.5178988e+00  2.2689254e+00  3.6674364e+00 ... -4.8334172e-01
   9.2667565e-02  5.8500001e-07]
 [ 1.3936870e-01 -4.5939609e-01  9.3638927e-01 ...  5.2074165e+00
   2.4532373e+00  3.6380001e-07]]


6.1 (Optional) The hidden code below can display the samples distribution. Uncomment them to acquire the distribution plot.

In [90]:
# X_all = np.vstack([X_train, X_test])

# y_train_1d = np.argmax(y_train, axis=1)
# y_test_1d  = np.argmax(y_test, axis=1)
# y_all_1d = np.hstack([y_train_1d, y_test_1d])

# D = pairwise_distances(X_all) 

In [91]:
# X_mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0).fit_transform(D)

In [92]:
# sns.scatterplot(x=X_mds[:,0], y=X_mds[:,1], hue=y_all_1d)
# plt.title("MDS – preserves original distances")

7. Now we have the train_ds and test_ds for training

In [93]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test))

# Training Area

**Could modify the model as you wish here. Again, in default, influenciae relies on TensorFlow, so use the TensorFlow model if you only want to change the model. Remember: Store the InfluenceModel into the model_list with the loss function. The InfluenceModel will be used to obtain influence later. If you don't change the estimation methods, then the final output at this step shall always be the model_list**

**Input**:Train and Test Set from Data Construction Section   
**Output**: Model List  
**Guideline**: Input -> Define the Model and Hyperparameters -> Train the Model -> Output

Always remember to train the model, get the influence model and store that in model list, unless you wish to change the estimation methods.

The default code now use the train and test set generated from the last section to train the model. The default hyperparameters are: 300 Epochs, Simple FeedForward Neural Network, CategoricalCrossEntropy Loss function, SGD optimizer. Within each epoch, the current model will be turned into an Influence Model and stored inside a model list. After the training, the model list will be passed to next section for influence estimation.

**One thing to be mentioned here is that some datasets may face high loss when changing the balance of classes. That means a more imbalanced set needs more epochs to reach high accuracy, while a more balanced set can be overfit under such epochs. Therefore, we add one mechanism to check if the model is trained to a satisfactory accuracy. By setting target accuracy, you can make sure everytime, the model is trained to a similar level. The allowance can help avoid some of the training fluctuation.**

In [94]:
from tensorflow.keras.regularizers import l2

1. **Could modify the model as you wish here as long as it is tensorflow.** Just remember: Store the InfluenceModel into the model_list with the loss function

In [95]:
seed_value = 42
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

model = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),  
    BatchNormalization(momentum=0.9),
    Dropout(0.0),
    Dense(8, activation='relu'),
    Dense(y_train.shape[1])
])
loss_fn = CategoricalCrossentropy(from_logits=True)
optimizer = SGD(learning_rate=0.001, momentum=0.9)
model.compile(loss=loss_fn, optimizer=optimizer, metrics=['accuracy'])

epochs = 300
unreduced_loss_fn = CategoricalCrossentropy(from_logits=True, reduction=tf.keras.losses.Reduction.NONE)
model_list = []
initial_model = tf.keras.models.clone_model(model)
initial_model.set_weights(model.get_weights())
model_list.append(InfluenceModel(initial_model, start_layer=-1, loss_function=unreduced_loss_fn))
for i in range(epochs):
  model.fit(train_ds.batch(256), epochs=1, validation_data=test_ds.batch(256), verbose=2)
  checkpoint_model = tf.keras.models.clone_model(model)
  checkpoint_model.set_weights(model.get_weights())
  model_list.append(InfluenceModel(checkpoint_model, start_layer=-1, loss_function=unreduced_loss_fn))
base_loss, acc = model.evaluate(test_ds.batch(32), verbose=2)
print(base_loss)

32/32 - 1s - loss: 0.8115 - accuracy: 0.6556 - val_loss: 0.8620 - val_accuracy: 0.4980 - 573ms/epoch - 18ms/step
32/32 - 0s - loss: 0.6348 - accuracy: 0.6999 - val_loss: 0.6786 - val_accuracy: 0.5900 - 72ms/epoch - 2ms/step
32/32 - 0s - loss: 0.5472 - accuracy: 0.7555 - val_loss: 0.5958 - val_accuracy: 0.6700 - 81ms/epoch - 3ms/step
32/32 - 0s - loss: 0.4967 - accuracy: 0.7980 - val_loss: 0.5478 - val_accuracy: 0.7380 - 75ms/epoch - 2ms/step
32/32 - 0s - loss: 0.4600 - accuracy: 0.8230 - val_loss: 0.5138 - val_accuracy: 0.7680 - 62ms/epoch - 2ms/step
32/32 - 0s - loss: 0.4300 - accuracy: 0.8418 - val_loss: 0.4856 - val_accuracy: 0.7900 - 69ms/epoch - 2ms/step
32/32 - 0s - loss: 0.4034 - accuracy: 0.8562 - val_loss: 0.4601 - val_accuracy: 0.8120 - 77ms/epoch - 2ms/step
32/32 - 0s - loss: 0.3795 - accuracy: 0.8669 - val_loss: 0.4367 - val_accuracy: 0.8200 - 74ms/epoch - 2ms/step
32/32 - 0s - loss: 0.3576 - accuracy: 0.8776 - val_loss: 0.4144 - val_accuracy: 0.8360 - 76ms/epoch - 2ms/step

In [96]:
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [97]:
train_logits = model.predict(
    X_train,
    batch_size=256,
    verbose=0
)

per_sample_loss_fn = tf.keras.losses.CategoricalCrossentropy(
    from_logits=True,
    reduction=tf.keras.losses.Reduction.NONE
)

train_losses = per_sample_loss_fn(
    y_train,
    train_logits
).numpy()

# Recover the original training IDs and labels
loss_df = pd.DataFrame({
    "Train_ID": train_ids,
    "Training_Loss": train_losses
})

In [98]:
loss_df.to_csv("loss_9_1_Run1.csv",index = False)

# Influence Estimation Area

**Again, you could use other influence analysis methods rather than IF/TC. You can also use any other Influence Function or TracIn implementation. Just Remember: 1. Make sure the package is unform throughout the framework. 2. Generate a Ranked influence list for each Influence Function and TracIn; Only the ranked influence list could be fed into the following analysis code.**

**The default code now use the model list, train set and test set to estimate the influence, and produce a ranked influence list for both IF and TC. The results are then saved in the root directory.**

**Input**:Model list from Training section, Train and Test Set from Data Construction Section   
**Output**: Two ranked Influence Lists for IF and TC.  
**Guideline**: Input -> Influence Estimation Methods -> Influence Matrix -> Output

1. Influence Function: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use influence_matrix.

In [99]:
train_ids = []
test_ids = []
train_samples_np = np.array([x.numpy() for x, y in train_ds])
train_ids = [round(sample[-1] * 1e10) for sample in train_samples_np]

In [100]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

influence_model = model_list[-1]
ihvp_calculator = ExactIHVP(influence_model, train_ds.batch(16))
influence_calculator = FirstOrderInfluenceCalculator(influence_model, train_ds, ihvp_calculator)

influence_matrix = np.zeros((num_test_samples, num_train_samples))

samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            influence_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(influence_matrix, axis=0).reshape(1, -1)
df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(df)

      Train_ID     Score
0         5018  0.011021
1        10999  0.001316
2         3146  0.002578
3         9147  0.001131
4        12955  0.004162
...        ...       ...
7995     13382  0.000015
7996     13339  0.001031
7997     15658  0.002239
7998     12979  0.000322
7999      9910  0.013742

[8000 rows x 2 columns]


2. TracIn: Here we use the influenciae package. The following code will directly generate the influence list. If you wish to have the matrix, just use TracIn_matrix.

In [101]:
num_test_samples = len(test_ds)
num_train_samples = len(train_ids)
test_ids = []

TracIn_matrix = np.zeros((num_test_samples, num_train_samples))
influence_calculator = TracIn(
    model_list, 0.001
)
samples_to_explain = test_ds.take(num_test_samples).batch(1)
explanation_ds = influence_calculator.top_k(samples_to_explain, train_ds.batch(16), k=num_train_samples, order=ORDER.DESCENDING)
for test_idx,((sample, label), top_k_values, top_k_samples) in enumerate(explanation_ds.as_numpy_iterator()):
    test_sample_id = round(sample[0][-1] * 1e10)
    test_ids.append(test_sample_id)
    influential_ids = [round(s[-1] * 1e10) for s in top_k_samples[0]] 
    influence_scores = top_k_values[0]
    id_to_index = {train_id: idx for idx, train_id in enumerate(train_ids)}
    for inf_id, score in zip(influential_ids, influence_scores):
        if inf_id in id_to_index:
            TracIn_matrix[test_idx, id_to_index[inf_id]] = score

flattened_row = np.median(TracIn_matrix, axis=0).reshape(1, -1)
TracIn_df = pd.DataFrame({'Train_ID': train_ids, 'Score': flattened_row.flatten()})
print(TracIn_df)

      Train_ID     Score
0         5018  0.003553
1        10999  0.004192
2         3146  0.003366
3         9147 -0.002018
4        12955  0.002122
...        ...       ...
7995     13382 -0.000055
7996     13339 -0.000595
7997     15658 -0.001433
7998     12979 -0.002374
7999      9910  0.007968

[8000 rows x 2 columns]


3. Here we turn both influence lists to the ranked influence lists and then store them for further processing.

In [102]:
df_sorted = df.sort_values(by="Score", ascending=False).reset_index(drop=True)

TracIn_sorted = TracIn_df.sort_values(by="Score", ascending=False).reset_index(drop=True)

In [103]:
TracIn_sorted.to_csv("TC_9_1_Run1.csv",index = False)
df_sorted.to_csv("IF_9_1_Run1.csv",index = False)

In [104]:
# print(df_sorted)

In [105]:
# train_df[train_df['id'] == 14825]